In [2]:
import os, shutil, rasterio, sys, logging
sys.path.append('backend/app/')
from rasterio.features import shapes, rasterize
from rasterio.mask import mask
from shapely.geometry import shape
import geopandas as gpd, pandas as pd
import numpy as np, xarray as xr
from backend.app.services import flow_functions
from shapely.geometry import Polygon, MultiPolygon
# from hydromt_wflow import WflowModel
import rioxarray as rioxr
np.random.seed(42)
logging.basicConfig(level=logging.DEBUG)

In [5]:
# Functions
def keep_polygon(geom):
    if geom.geom_type == 'GeometryCollection':
        polys = [g for g in geom.geoms if isinstance(g, (Polygon, MultiPolygon))]
        if len(polys) == 0: return None
        return polys[0]
    return geom

def fix_invalid_polygon(gdf, cols):
    gdf_new, name = gdf.copy(), cols[0]
    gdf_valid, gdf_nan = gdf_new[gdf_new[name] != ''], gdf_new[gdf_new[name] == '']
    if gdf_nan.shape[0] > 0:
        gdf_valid['geometry'] = gdf_valid['geometry'].apply(keep_polygon)
        gdf_nan['geometry'] = gdf_nan['geometry'].apply(keep_polygon)
        # Spatial join nearest
        gdf_filled = gpd.sjoin_nearest(
            gdf_nan, gdf_valid[['geometry', name]], how='left', distance_col='dist'
        )
        gdf_filled = gdf_filled.drop_duplicates(subset='_id')
        gdf_new.loc[gdf_filled.index, cols] = gdf_valid.loc[gdf_filled['index_right'], cols].values
    gdf_new['geometry'] = gdf_new['geometry'].apply(keep_polygon)
    return gdf_new

def clip_catchment(catchment, raster_path, out_path, inside=False):
    with rasterio.open(raster_path) as src:
        geoms = catchment.geometry.values
        # Clip raster
        out_image, out_transform = mask(
            src, geoms, crop=False, nodata=-9999, invert=inside
        )
        out_meta = src.meta.copy()
    out_meta.update({
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform, "nodata": -9999
    })
    with rasterio.open(out_path, "w", **out_meta) as dest:
        dest.write(out_image)
    del dest

def write_tif(path, terrain, geo, col=''):
    transform = terrain.transform
    if col == '': shapes = ((geom, 1) for geom in geo.geometry)
    else: shapes = ((geom, value) for geom, value in zip(geo.geometry, geo[col]))
    raster = rasterize(
        shapes=shapes, out_shape=(terrain.height, terrain.width),
        transform=transform, fill=-9999, dtype="float32", all_touched=True
    )
    meta = {
        "driver": "GTiff", "height": terrain.height,
        "width": terrain.width, "count": 1, "dtype": "float32", 
        "crs": terrain.crs, "transform": transform, "nodata": -9999
    }
    with rasterio.open(path, "w", **meta) as dst:
        dst.write(raster, 1)
    del raster

In [31]:
folder = 'wflow_model'
catchment_path = os.path.join(folder, 'inputs', 'catchment.geojson')
terrain_path = os.path.join(folder, 'inputs', 'dtm10.tif')
soil_path = os.path.join(folder, 'inputs', 'soil.geojson')
land_path = os.path.join(folder, 'inputs', 'land.geojson')
river_path = os.path.join(folder, 'inputs', 'river.geojson')
weather_path = os.path.join(folder, 'inputs', 'alesund_weather.csv')
terrain = rasterio.open(terrain_path)
catchment = gpd.read_file(catchment_path)
initial_param = [0.25,0.3,50,0,0,500,0,0,100]
soil = gpd.read_file(soil_path)
land = gpd.read_file(land_path)
river = gpd.read_file(river_path)
weather = pd.read_csv(weather_path)

In [13]:
# Clip dtm to catchment
catchment_UTM = catchment.to_crs(terrain.crs)
terrain_out_path = os.path.normpath(os.path.join(folder, 'staticmaps', "dtm.tif"))
clip_catchment(catchment_UTM, terrain_path, terrain_out_path)

In [ ]:
# Process river
river_UTM = river.to_crs(terrain.crs)
cols = {
    'width': (0.05, 2), 'depth': (1, 5), 'manning_n': (0.03, 0.06)
}
river_cols = river_UTM.columns.drop('geometry', errors='ignore')
for col in river_cols:
    river_UTM[col] = pd.to_numeric(river_UTM[col], errors='coerce')
for col, (low, high) in cols.items():
    river_mask = river_UTM[col].isna() | (river_UTM[col] == 'None')
    river_UTM.loc[river_mask, col] = np.round(np.random.uniform(low, high, river_mask.sum()), 3)
river_dict = {
    'river': '', 'river_width': 'width', 'river_depth': 'depth', 'river_n': 'manning_n'
}
for key, value in river_dict.items():
    river_path = os.path.normpath(os.path.join(folder, 'staticmaps', f'{key}.tif'))
    write_tif(river_path, terrain, river_UTM, value)


In [6]:
# Fix invalid soil polygon
soil_UTM = soil.to_crs(terrain.crs)
soil_cols = ['soil', 'theta_s', 'theta_r', 'k_sat_ver', 'soil_depth', 'conductivity_decay', 'brooks_corey']
soil_UTM = fix_invalid_polygon(soil_UTM, soil_cols)
soil_UTM = soil_UTM[soil_UTM.soil != 'Water']
# soil_out_path = os.path.normpath(os.path.join(folder, 'inputs', "soil.geojson"))
# soil_UTM.to_file(soil_out_path, driver='GeoJSON', encoding='utf-8')
soil_layers = ['theta_s', 'theta_r', 'k_sat_ver', 'soil_depth', 'conductivity_decay', 'brooks_corey']
for value in soil_layers:
    soil_path = os.path.normpath(os.path.join(folder, 'staticmaps', f'{value}.tif'))
    write_tif(soil_path, terrain, soil_UTM, value)

In [59]:
# Fix invalid land polygon
land_UTM = land.to_crs(terrain.crs)
land_cols = ['land', 'LAI', 'root_depth', 'interception', 'manning_n', 'albedo', 'kc']
land_UTM = fix_invalid_polygon(land_UTM, land_cols)
lookup = land_UTM.groupby("land").agg({
    "LAI": "first", "root_depth": "first", "interception": "first",
    "manning_n": "first", "albedo": "first", "kc": "first"
}).reset_index()
for item in lookup['land'].values:
    df = land_UTM[land_UTM['land'] == item]
    if len(df) > 1:
        land_UTM.loc[df.index, 'id'] = str(lookup[lookup['land'] == item].index.start)
land_UTM['id'] = land_UTM['id'].astype(int)
land_path = os.path.normpath(os.path.join(folder, 'staticmaps', 'land.tif'))
write_tif(land_path, terrain, land_UTM, 'id')
lookup = lookup.drop('land', axis=1)
lookup.insert(0, 'class_id', lookup.index)
lookup.reset_index(drop=True, inplace=True)

# land_out_path = os.path.normpath(os.path.join(folder, 'inputs', "land.geojson"))
# land_UTM.to_file(land_out_path, driver='GeoJSON', encoding='utf-8')
land_layers = ['LAI', 'root_depth', 'interception', 'manning_n', 'albedo', 'kc']
for value in land_layers:
    land_path = os.path.normpath(os.path.join(folder, 'staticmaps', f'{value}.tif'))
    write_tif(land_path, terrain, land_UTM, value)

In [38]:
print(lookup)

   class_id  LAI  root_depth  interception  manning_n  albedo    kc
0         0  0.1         0.1           0.2       0.02    0.25  0.20
1         1  5.0         1.5           3.0       0.40    0.13  1.10
2         2  0.5         0.1           0.5       0.05    0.15  0.30
3         3  2.0         0.5           1.0       0.15    0.23  0.90
4         4  0.0         0.0           0.0       0.03    0.80  0.10
5         5  0.0         0.0           0.0       0.03    0.07  1.05


In [30]:
land_UTM

,id,_id,land,LAI,root_depth,interception,manning_n,albedo,kc,geometry
0,3,1,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((55837.796 6954963.694, 55836.605 695..."
1,2,2,Impervious/Urban,0.5,0.1,0.5,0.05,0.15,0.3,"POLYGON ((55871.879 6954959.108, 55870.688 695..."
2,3,3,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((55866.254 6954968.88, 55863.871 6954..."
3,3,4,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((55859.437 6954969.797, 55858.246 695..."
4,3,5,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((55838.988 6954972.548, 55837.796 695..."
...,...,...,...,...,...,...,...,...,...,...
8832,2,8833,Impervious/Urban,0.5,0.1,0.5,0.05,0.15,0.3,"POLYGON ((60967.119 6957042.192, 60965.94 6957..."
8833,3,8834,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((60848.007 6957085.199, 60846.828 695..."
8834,3,8835,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((60957.352 6957125.056, 60956.173 695..."
8835,3,8836,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((61571.585 6957206.475, 61570.408 695..."


In [4]:
weather['datetime'] = pd.to_datetime(weather['datetime'])
weather

,datetime,precip_mm,temp_C,shortwave_Wm2,longwave_Wm2,wind_mps,relhum_pct,pressure
0,2025-01-01 00:00:00,2.447,5.172,0.000,320.125,7.620,86.733,1009.946
1,2025-01-01 01:00:00,2.383,5.172,129.410,384.656,7.993,74.095,987.605
2,2025-01-01 02:00:00,3.192,5.172,250.000,384.239,8.049,70.175,997.114
3,2025-01-01 03:00:00,0.759,5.172,353.553,364.490,7.289,87.534,988.409
4,2025-01-01 04:00:00,1.788,5.172,433.013,322.710,8.959,91.852,1013.289
...,...,...,...,...,...,...,...,...
8755,2025-12-31 19:00:00,1.529,5.000,0.000,343.680,5.803,71.896,990.648
8756,2025-12-31 20:00:00,0.724,5.000,0.000,301.795,9.012,93.424,1017.078
8757,2025-12-31 21:00:00,0.415,5.000,0.000,378.202,9.810,85.269,1017.871
8758,2025-12-31 22:00:00,1.135,5.000,0.000,360.407,9.215,84.581,1012.472


In [44]:
type(weather['datetime'].values[0])

numpy.datetime64

In [ ]:
data_des = os.path.join(folder, 'staticmaps')
# dem = rioxr.open_rasterio(os.path.join(data_des, 'dtm.tif')).squeeze()
# thetaS = rioxr.open_rasterio(os.path.join(data_des, 'theta_s.tif')).squeeze()
# thetaR = rioxr.open_rasterio(os.path.join(data_des, 'theta_r.tif')).squeeze()
# KsatVer = rioxr.open_rasterio(os.path.join(data_des, 'k_sat_ver.tif')).squeeze()
# SoilThickness = rioxr.open_rasterio(os.path.join(data_des, 'soil_depth.tif')).squeeze()
# M = rioxr.open_rasterio(os.path.join(data_des, 'conductivity_decay.tif')).squeeze()
# c = rioxr.open_rasterio(os.path.join(data_des, 'brooks_corey.tif')).squeeze()
# landUse = rioxr.open_rasterio(os.path.join(data_des, 'land.tif')).astype('int32').squeeze()
# ds = xr.Dataset({
#     'elevation': dem, 'theta_s': thetaS, 'theta_r': thetaR, 'ksat': KsatVer,
#     'soil_thickness': SoilThickness, 'm': M, 'c': c, 'landuse': landUse
# })
# mod.set_grid(ds)

In [71]:
mod = WflowModel(root='model', mode='w', config_fn=os.path.join(folder, 'inputs', 'config.yaml'))
mod.build()

INFO:hydromt_wflow.wflow:Initializing wflow model from hydromt_wflow (v0.8.0).
INFO:hydromt_wflow.wflow:Parsing data catalog from c:\Envs\hyd_ai\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
INFO:hydromt_wflow.wflow:Write model data to c:\Users\vanln\Downloads\Hydro-AI-Platform\model
DEBUG:hydromt_wflow.wflow:Default config read from c:\Envs\hyd_ai\Lib\site-packages\hydromt_wflow\data\wflow\wflow_sbm.toml
INFO:hydromt_wflow.wflow:Writing model config to c:\Users\vanln\Downloads\Hydro-AI-Platform\model\wflow_model\inputs\config.yaml


In [63]:
print(mod.grid)

<xarray.Dataset> Size: 0B
Dimensions:  ()
Data variables:
    *empty*
